## ℹ️ Provenance

This notebook is **agent-assisted but human-controlled**. The complete
history of every modification — what changed, when, who made it, and
why — is in:

- `notebook-build/CHANGELOG.md`  — human-readable history
- `notebook-build/build_manifest.json` — machine-readable build record
  with SHA-256 fingerprints of every step script
- `notebook-build/0X_*.py`  — the actual code that produced the change
- `linda-example.original-20260421.ipynb` — immutable seed (the
  template author's original notebook), preserved as the base of every
  rebuild
- `_notebook_backups/` — automatic timestamped backups created before
  every rebuild

**To reproduce or audit this notebook from scratch:**

```bash
cd LINDA-STUFF/notebook-build
python build.py --validate-only        # check the current notebook
python build.py                        # rebuild from the seed + scripts
python build.py --list                 # show every step + sha
```

If you edit the notebook directly (without going through the build
pipeline), please add an `EDITED` line to `CHANGELOG.md` so future
reviewers can tell agent-produced cells from human-edited ones.

# Lesion segmentation & visualization with LINDA

A flexible, reusable workflow for:

1. Pulling or pointing at a structural MRI dataset (BIDS or flat folder).
2. Running LINDA lesion segmentation on one subject or many.
3. Visualizing lesions and atlas overlaps interactively.
4. Computing per-subject and group-level lesion statistics.

**Author of this template**: Michèle Masson-Trottier

#### Tool citation
Pustina, D., Coslett, H. B., Turkeltaub, P. E., Tustison, N., Schwartz, M. F., & Avants, B. (2016).
Automated segmentation of chronic stroke lesions using LINDA: Lesion identification with neighborhood data analysis.
*Hum Brain Mapp*, 37(4), 1405–1421. https://doi.org/10.1002/hbm.23110


## How to use this notebook

The whole workflow is driven by a single configuration cell (`CONFIG` below).
Change values there once, and every other cell will pick them up automatically.

You can run it on:

- **An OpenNeuro dataset** (default: ds004884) — fetched with `datalad`.
- **A local BIDS dataset** — point `DATASET_SOURCE` at a folder.
- **A flat folder of T1w images** — works without `sub-*/ses-*` structure.
- **A single T1w file** — for a quick one-off segmentation.

If you only want to *look* at lesions you already have, set `RUN_SEGMENTATION = False`.


## Load software tools and import python libraries

In [ ]:
# Load LINDA via the Neurodesk module system
import module
await module.load('linda/0.5.1')
await module.list()


In [ ]:
%%capture
!pip install pydicom mystmd nilearn


In [ ]:
# Standard imports used throughout the notebook
import os
import re
import shutil
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
from nilearn import datasets, image, plotting


## Configuration

In [ ]:
# ============================================================
# CONFIGURATION — single source of truth for the whole notebook
# ============================================================
# Edit values here and re-run all cells. Every downstream cell
# reads from this CONFIG dict, so you do not need to edit them.
# ------------------------------------------------------------

CONFIG = {
    # --- Where the notebook stores its working files ----------
    # PROJECT_DIR is the root for everything the notebook creates.
    # Defaults to the folder you launched the notebook from.
    "PROJECT_DIR": Path.cwd(),

    # --- Dataset source ---------------------------------------
    # One of:
    #   "openneuro"   – fetch by accession ID with datalad
    #   "local"       – use a local BIDS dataset
    #   "flat"        – a folder of *_T1w.nii.gz files (no sub-/ses- needed)
    #   "single"      – a single T1w file
    "DATASET_SOURCE":  "openneuro",
    "OPENNEURO_ID":    "ds004884",          # used when DATASET_SOURCE == "openneuro"
    "OPENNEURO_TAG":   "1.0.2",             # git tag/branch to checkout (None = HEAD)
    "LOCAL_DATASET":   None,                # Path to local BIDS or flat folder
    "SINGLE_T1W":      None,                # Path to a single .nii.gz T1w file

    # --- Subject filter ---------------------------------------
    # Limit which subjects to process. None = process everything found.
    # You can pass a list like ["sub-M2040", "sub-M2041"], or a regex string.
    "SUBJECT_FILTER":  None,
    "SESSION_FILTER":  None,
    # If a dataset has multiple T1w runs per session, take the first by default.
    # Set to "all" to process every T1w found.
    "T1W_SELECTION":   "first",

    # --- Where outputs go -------------------------------------
    # Anatomical images are symlinked into ANAT_DIR (so we don't
    # touch the raw dataset). LINDA writes into DERIV_DIR.
    # Default matches the original notebook's folder so existing
    # runs of ds004884 are picked up automatically.
    "ANAT_SUBDIR":         "ds004884_anat",       # under PROJECT_DIR
    "DERIVATIVES_SUBDIR":  "derivatives/lesion_mask",  # under ANAT_SUBDIR

    # --- Atlas options ----------------------------------------
    "ATLAS_SUBDIR":   "atlases",                  # under PROJECT_DIR
    "ATLASES":        ["HarvardOxford", "AAL", "Destrieux", "Schaefer400"],
    "DEFAULT_ATLAS":  "HarvardOxford",            # used by single-subject views

    # --- Behaviour --------------------------------------------
    "RUN_SEGMENTATION":   True,                   # set False to only visualize
    "LESION_THRESHOLD":   0.5,                    # binarize lesion maps at this prob
    "OVERWRITE":          False,                  # re-run LINDA even if output exists
    "FORCE_FETCH":        False,                  # re-ingest raw dataset even if ANAT_DIR is populated
    "VERBOSE":            True,
}

# ------------------------------------------------------------
# Derived paths — do not normally need to be edited
# ------------------------------------------------------------
PROJECT_DIR     = Path(CONFIG["PROJECT_DIR"]).resolve()
ANAT_DIR        = PROJECT_DIR / CONFIG["ANAT_SUBDIR"]
DERIV_DIR       = ANAT_DIR / CONFIG["DERIVATIVES_SUBDIR"]
ATLAS_DIR       = PROJECT_DIR / CONFIG["ATLAS_SUBDIR"]
RAW_DATASET_DIR = PROJECT_DIR / (CONFIG.get("OPENNEURO_ID") or "raw_dataset")

for d in (ANAT_DIR, DERIV_DIR, ATLAS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Export to the shell environment so %%bash cells can read them
os.environ["PROJECT_DIR"]     = str(PROJECT_DIR)
os.environ["ANAT_DIR"]        = str(ANAT_DIR)
os.environ["DERIV_DIR"]       = str(DERIV_DIR)
os.environ["ATLAS_DIR"]       = str(ATLAS_DIR)
os.environ["RAW_DATASET_DIR"] = str(RAW_DATASET_DIR)
os.environ["NILEARN_DATA"]    = str(ATLAS_DIR)
os.environ["DATASET_SOURCE"]  = CONFIG["DATASET_SOURCE"]
os.environ["OPENNEURO_ID"]    = CONFIG.get("OPENNEURO_ID") or ""
os.environ["OPENNEURO_TAG"]   = CONFIG.get("OPENNEURO_TAG") or ""
os.environ["LOCAL_DATASET"]   = str(CONFIG["LOCAL_DATASET"] or "")
os.environ["SINGLE_T1W"]      = str(CONFIG["SINGLE_T1W"] or "")
os.environ["FORCE_FETCH"]     = "1" if CONFIG["FORCE_FETCH"] else "0"

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k:18s} = {v}")
print()
print("Derived paths:")
print(f"  PROJECT_DIR     = {PROJECT_DIR}")
print(f"  ANAT_DIR        = {ANAT_DIR}")
print(f"  DERIV_DIR       = {DERIV_DIR}")
print(f"  ATLAS_DIR       = {ATLAS_DIR}")
print(f"  RAW_DATASET_DIR = {RAW_DATASET_DIR}")


### Load atlases

These atlases are used to compute regional overlap with each lesion mask.
They are cached under `ATLAS_DIR` so subsequent runs don't re-download.
Atlases are loaded into the `ALL_ATLASES` registry; downstream cells pick from it
based on `CONFIG["ATLASES"]`.


In [ ]:
# ============================================================
# Atlas loader — caches everything under ATLAS_DIR
# ============================================================
print(f"Atlas directory: {ATLAS_DIR}")

class SimpleAtlas:
    """Tiny container so AAL (and any custom atlas) looks like a nilearn atlas."""
    def __init__(self, maps, labels):
        self.maps = maps
        self.labels = labels


def _fetch_harvard_oxford():
    a = datasets.fetch_atlas_harvard_oxford(
        "cort-maxprob-thr25-2mm", data_dir=str(ATLAS_DIR)
    )
    return a


def _fetch_destrieux():
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        return datasets.fetch_atlas_destrieux_2009(data_dir=str(ATLAS_DIR))


def _fetch_schaefer400():
    return datasets.fetch_atlas_schaefer_2018(
        n_rois=400, yeo_networks=7, data_dir=str(ATLAS_DIR)
    )


def _fetch_aal():
    """nilearn fetch with a fallback to a public mirror if SSL fails."""
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", DeprecationWarning)
            obj = datasets.fetch_atlas_aal(version="SPM12", data_dir=str(ATLAS_DIR))
        return SimpleAtlas(nib.load(obj.maps), obj.labels)
    except Exception as e:
        print(f"  ⚠ nilearn AAL fetch failed ({e}); using mirror")
        import requests
        import xml.etree.ElementTree as ET

        d = ATLAS_DIR / "aal_mirror"
        d.mkdir(exist_ok=True)
        nii = d / "AAL.nii.gz"
        xml = d / "AAL.xml"
        BASE = ("https://raw.githubusercontent.com/Neurita/std_brains/"
                "master/atlases/aal_SPM12/aal/atlas")

        def _dl(url, dest):
            if dest.exists():
                return
            r = requests.get(url, stream=True, timeout=60)
            r.raise_for_status()
            with open(dest, "wb") as fh:
                for chunk in r.iter_content(1024 * 1024):
                    if chunk:
                        fh.write(chunk)

        _dl(f"{BASE}/AAL.nii.gz", nii)
        _dl(f"{BASE}/AAL.xml", xml)
        labels = ["Background"]
        try:
            for elem in ET.parse(xml).getroot().iter():
                if elem.text and elem.text.strip().isalpha():
                    labels.append(elem.text.strip())
        except Exception:
            pass
        return SimpleAtlas(nib.load(str(nii)), labels)


_FETCHERS = {
    "HarvardOxford": _fetch_harvard_oxford,
    "Destrieux":     _fetch_destrieux,
    "Schaefer400":   _fetch_schaefer400,
    "AAL":           _fetch_aal,
}

ALL_ATLASES = {}
for name in CONFIG["ATLASES"]:
    if name not in _FETCHERS:
        print(f"  ⚠ Unknown atlas '{name}' — skipped")
        continue
    print(f"• Fetching {name}")
    try:
        ALL_ATLASES[name] = _FETCHERS[name]()
        n = len(ALL_ATLASES[name].labels)
        print(f"  ✓ {n} regions")
    except Exception as e:
        print(f"  ❌ Failed: {e}")

print("\nAtlases ready:", list(ALL_ATLASES.keys()))


## Data preparation

This section either fetches the dataset (OpenNeuro / datalad) or registers an
existing local dataset, and discovers the T1w images to process. Subjects/sessions
can be filtered through `CONFIG["SUBJECT_FILTER"]` and `CONFIG["SESSION_FILTER"]`.


In [ ]:
%%bash
# ============================================================
# Fetch / register the source dataset
# - Skips entirely when ANAT_DIR already contains T1w files
#   (unless FORCE_FETCH=1)
# - Uses symlinks rather than copies for speed / no disk waste
# ============================================================
set -uo pipefail
shopt -s nullglob

# ---- 0. Fast path: ANAT_DIR already populated? --------------
existing=$(find "$ANAT_DIR" -maxdepth 6 -name "*_T1w.nii.gz" 2>/dev/null | wc -l)
if [ "$existing" -gt 0 ] && [ "${FORCE_FETCH:-0}" != "1" ]; then
    echo "✓ ANAT_DIR already has $existing T1w file(s) — skipping fetch."
    echo "   ANAT_DIR=$ANAT_DIR"
    echo "   (set CONFIG[\"FORCE_FETCH\"]=True in the CONFIG cell to re-fetch)"
    find "$ANAT_DIR" -maxdepth 6 -name "*_T1w.nii.gz" | head -n 20
    echo "(showing first 20)"
    exit 0
fi

case "$DATASET_SOURCE" in
  openneuro)
      if [ -z "$OPENNEURO_ID" ]; then
          echo "❌ DATASET_SOURCE=openneuro but OPENNEURO_ID is empty"; exit 1;
      fi
      if [ ! -d "$RAW_DATASET_DIR" ]; then
          echo "📥 datalad install $OPENNEURO_ID → $RAW_DATASET_DIR"
          datalad install "https://github.com/OpenNeuroDatasets/${OPENNEURO_ID}.git" \
              -p "$RAW_DATASET_DIR"
          if [ -n "$OPENNEURO_TAG" ]; then
              ( cd "$RAW_DATASET_DIR" && git checkout "$OPENNEURO_TAG" )
          fi
      else
          echo "✓ Using existing dataset at $RAW_DATASET_DIR"
      fi

      echo "📂 Linking anatomicals into $ANAT_DIR (first session per subject)"
      cd "$RAW_DATASET_DIR"
      n_subj=0; n_t1=0
      for subj in sub-*; do
          [ -d "$subj" ] || continue
          n_subj=$((n_subj + 1))
          first_ses=$(ls -d "$subj"/ses-* 2>/dev/null | head -n 1)
          if [ -n "$first_ses" ]; then
              mkdir -p "$ANAT_DIR/$first_ses/anat"
              for t1 in "$first_ses"/anat/*_T1w.nii.gz; do
                  [ -e "$t1" ] || continue
                  # fetch annex content on demand, then symlink
                  datalad get "$t1" >/dev/null 2>&1 || true
                  ln -sfn "$(realpath "$t1")" \
                      "$ANAT_DIR/$first_ses/anat/$(basename "$t1")"
                  n_t1=$((n_t1 + 1))
              done
          elif [ -d "$subj/anat" ]; then
              mkdir -p "$ANAT_DIR/$subj/anat"
              for t1 in "$subj/anat"/*_T1w.nii.gz; do
                  [ -e "$t1" ] || continue
                  datalad get "$t1" >/dev/null 2>&1 || true
                  ln -sfn "$(realpath "$t1")" \
                      "$ANAT_DIR/$subj/anat/$(basename "$t1")"
                  n_t1=$((n_t1 + 1))
              done
          fi
      done
      echo "✓ Linked $n_t1 T1w file(s) from $n_subj subject(s)"
      ;;

  local)
      if [ -z "$LOCAL_DATASET" ] || [ ! -d "$LOCAL_DATASET" ]; then
          echo "❌ LOCAL_DATASET is not a directory: $LOCAL_DATASET"; exit 1;
      fi
      echo "✓ Using local dataset at $LOCAL_DATASET"
      cd "$LOCAL_DATASET"
      n_t1=0
      for subj in sub-*; do
          [ -d "$subj" ] || continue
          for ses in "$subj"/ses-*; do
              [ -d "$ses" ] || continue
              mkdir -p "$ANAT_DIR/$ses/anat"
              for t1 in "$ses"/anat/*_T1w.nii.gz; do
                  [ -e "$t1" ] || continue
                  ln -sfn "$(realpath "$t1")" \
                      "$ANAT_DIR/$ses/anat/$(basename "$t1")"
                  n_t1=$((n_t1 + 1))
              done
          done
          if [ -d "$subj/anat" ]; then
              mkdir -p "$ANAT_DIR/$subj/anat"
              for t1 in "$subj/anat"/*_T1w.nii.gz; do
                  [ -e "$t1" ] || continue
                  ln -sfn "$(realpath "$t1")" \
                      "$ANAT_DIR/$subj/anat/$(basename "$t1")"
                  n_t1=$((n_t1 + 1))
              done
          fi
      done
      echo "✓ Linked $n_t1 T1w file(s)"
      ;;

  flat)
      if [ -z "$LOCAL_DATASET" ] || [ ! -d "$LOCAL_DATASET" ]; then
          echo "❌ LOCAL_DATASET is not a directory: $LOCAL_DATASET"; exit 1;
      fi
      echo "📂 Indexing flat folder $LOCAL_DATASET"
      mkdir -p "$ANAT_DIR/flat/anat"
      n_t1=0
      for t1 in "$LOCAL_DATASET"/*_T1w.nii.gz; do
          [ -e "$t1" ] || continue
          ln -sfn "$(realpath "$t1")" "$ANAT_DIR/flat/anat/$(basename "$t1")"
          n_t1=$((n_t1 + 1))
      done
      echo "✓ Linked $n_t1 T1w file(s)"
      ;;

  single)
      if [ -z "$SINGLE_T1W" ] || [ ! -f "$SINGLE_T1W" ]; then
          echo "❌ SINGLE_T1W is not a file: $SINGLE_T1W"; exit 1;
      fi
      mkdir -p "$ANAT_DIR/single/anat"
      ln -sfn "$(realpath "$SINGLE_T1W")" \
          "$ANAT_DIR/single/anat/$(basename "$SINGLE_T1W")"
      echo "✓ Registered 1 T1w file"
      ;;

  *)
      echo "❌ Unknown DATASET_SOURCE: $DATASET_SOURCE"; exit 1;
      ;;
esac

echo
echo "=== T1w files now visible under ANAT_DIR ==="
find "$ANAT_DIR" -maxdepth 6 -name "*_T1w.nii.gz" | head -n 20
echo "(showing first 20)"


In [ ]:
# ============================================================
# Discover T1w images to process — BIDS-aware but BIDS-agnostic
# ============================================================
SUB_RE = re.compile(r"(sub-[A-Za-z0-9]+)")
SES_RE = re.compile(r"(ses-[A-Za-z0-9]+)")


def _matches_filter(value, flt):
    """Apply CONFIG filter — list of strings, single string, regex, or None."""
    if flt is None:
        return True
    if isinstance(flt, (list, tuple, set)):
        return value in flt
    if isinstance(flt, str):
        # treat as regex if it contains regex meta-chars, else equality
        if any(c in flt for c in ".*+?[](){}|^$\\"):
            return re.search(flt, value) is not None
        return value == flt
    return True


def discover_t1w(root: Path, sub_filter=None, ses_filter=None, selection="first"):
    """
    Return a list of dicts:
      {"subject": "sub-XXX", "session": "ses-YYY" or None, "t1w": Path}

    Works with:
      - sub-*/ses-*/anat/*_T1w.nii.gz   (standard BIDS w/ sessions)
      - sub-*/anat/*_T1w.nii.gz         (BIDS without sessions)
      - flat folder of *_T1w.nii.gz
    Subject/session IDs are taken from path components first,
    falling back to filename parsing.
    """
    root = Path(root)
    found = []
    for t1 in sorted(root.rglob("*_T1w.nii.gz")):
        rel = t1.relative_to(root)
        parts = rel.parts
        sub = next((p for p in parts if p.startswith("sub-")), None)
        ses = next((p for p in parts if p.startswith("ses-")), None)
        if sub is None:
            m = SUB_RE.search(t1.name)
            sub = m.group(1) if m else f"sub-{t1.stem.replace('.nii','')}"
        if ses is None:
            m = SES_RE.search(t1.name)
            ses = m.group(1) if m else None
        if not _matches_filter(sub, sub_filter):
            continue
        if ses is not None and not _matches_filter(ses, ses_filter):
            continue
        found.append({"subject": sub, "session": ses, "t1w": t1})

    if selection == "first":
        seen = {}
        for entry in found:
            key = (entry["subject"], entry["session"])
            seen.setdefault(key, entry)
        found = list(seen.values())

    return found


SUBJECTS = discover_t1w(
    ANAT_DIR,
    sub_filter=CONFIG["SUBJECT_FILTER"],
    ses_filter=CONFIG["SESSION_FILTER"],
    selection=CONFIG["T1W_SELECTION"],
)

print(f"Discovered {len(SUBJECTS)} T1w image(s) for processing.")
for s in SUBJECTS[:10]:
    print(f"  • {s['subject']}  {s['session'] or '(no session)':<12s}  {s['t1w'].name}")
if len(SUBJECTS) > 10:
    print(f"  … and {len(SUBJECTS) - 10} more")


def deriv_path_for(entry):
    """Where LINDA outputs for one entry should live (BIDS-derivatives style)."""
    sub = entry["subject"]
    ses = entry["session"]
    if ses:
        return DERIV_DIR / sub / ses / "anat" / "linda"
    return DERIV_DIR / sub / "anat" / "linda"


def lesion_mni_for(entry):
    """Convenience: where Lesion_in_MNI.nii.gz lives for an entry."""
    return deriv_path_for(entry) / "Lesion_in_MNI.nii.gz"


## Lesion segmentation

LINDA writes its outputs into a per-subject folder under `DERIV_DIR`. Existing
outputs are skipped unless `CONFIG["OVERWRITE"]` is `True`. Set
`CONFIG["RUN_SEGMENTATION"] = False` to skip this section entirely (e.g. when you
already have lesion masks and just want to visualize).


In [ ]:
# ============================================================
# Run LINDA on every discovered T1w
# ============================================================
import subprocess

if not CONFIG["RUN_SEGMENTATION"]:
    print("RUN_SEGMENTATION = False — skipping segmentation.")
else:
    print(f"Running LINDA on {len(SUBJECTS)} subject(s) "
          f"(overwrite={CONFIG['OVERWRITE']})")
    n_done, n_skipped, n_failed = 0, 0, 0

    for entry in SUBJECTS:
        out_dir = deriv_path_for(entry)
        out_dir.mkdir(parents=True, exist_ok=True)
        expected = out_dir / "Lesion_in_MNI.nii.gz"

        tag = f"{entry['subject']} {entry['session'] or ''}".strip()
        if expected.exists() and expected.stat().st_size > 0 and not CONFIG["OVERWRITE"]:
            print(f"  ✓ {tag}: already done — skipping")
            n_skipped += 1
            continue

        print(f"  ▶ {tag}: running LINDA on {entry['t1w'].name}")
        try:
            subprocess.run(
                ["linda_predict.sh", str(entry["t1w"]), str(out_dir)],
                check=True,
            )
            if expected.exists() and expected.stat().st_size > 0:
                print(f"    ✓ done")
                n_done += 1
            else:
                print(f"    ❌ LINDA finished but no Lesion_in_MNI.nii.gz")
                n_failed += 1
        except subprocess.CalledProcessError as e:
            print(f"    ❌ LINDA crashed: {e}")
            n_failed += 1

    print(f"\nSegmentation summary: {n_done} new, {n_skipped} skipped, "
          f"{n_failed} failed (out of {len(SUBJECTS)})")


### Relocate LINDA outputs

In some LINDA versions, the predictor writes a `linda/` folder *next to the input
T1w*. The cell below moves any such folders into the BIDS-derivatives layout used
by the rest of the notebook. If your LINDA run already wrote into `DERIV_DIR`,
this is a no-op.


In [ ]:
# ============================================================
# Move stray linda/ folders that LINDA may have written next to the
# input T1w into the canonical derivatives location.
# ============================================================
moved = 0
for t1 in ANAT_DIR.rglob("*_T1w.nii.gz"):
    src = t1.parent / "linda"
    if not src.is_dir():
        continue
    rel = t1.parent.relative_to(ANAT_DIR)  # sub-*/(ses-*/)anat
    dst = DERIV_DIR / rel / "linda"
    if dst.exists():
        continue
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(src), str(dst))
    print(f"  ↪ moved {src} → {dst}")
    moved += 1

if moved == 0:
    print("No stray linda/ folders to relocate — derivatives already in place.")
else:
    print(f"Relocated {moved} folder(s)")


In [ ]:
# Quick check: how many subjects have a Lesion_in_MNI.nii.gz?
have_lesion = []
missing = []
for entry in SUBJECTS:
    if lesion_mni_for(entry).exists():
        have_lesion.append(entry)
    else:
        missing.append(entry)

print(f"Lesion masks present: {len(have_lesion)}/{len(SUBJECTS)}")
if missing and CONFIG["VERBOSE"]:
    print("Missing for:")
    for e in missing[:10]:
        print(f"  - {e['subject']} {e['session'] or ''}")
    if len(missing) > 10:
        print(f"  … and {len(missing) - 10} more")


## Results & visualization

The cells below build on the `SUBJECTS` list and `CONFIG`. Each part is
independent — you can re-run the QC dashboard, the group overlap map, or the
atlas overlap analysis on its own.


### Unified QC dashboard

A single interactive viewer for inspecting LINDA outputs. The dropdown lets you
jump between subjects; the tabs switch between skull-stripping QC, lesion in
native space, lesion in MNI space, and the atlas overlay. Subjects without LINDA
output are omitted so the dashboard never crashes on missing files.


In [ ]:
# ============================================================
# Unified interactive QC dashboard
# Skull-strip / native lesion / MNI lesion / atlas overlay
# ============================================================
from ipyniivue import NiiVue
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output


def _exists(p):
    return p is not None and Path(p).exists()


# Only show subjects with at least Lesion_in_MNI.nii.gz
qc_subjects = [e for e in SUBJECTS if lesion_mni_for(e).exists()]
qc_labels = [
    f"{e['subject']}{' ' + e['session'] if e['session'] else ''}"
    for e in qc_subjects
]

if not qc_subjects:
    display(HTML("<p><em>No lesion masks found yet — run segmentation first.</em></p>"))
else:
    qc_output = widgets.Output()

    def _show_qc(label, view):
        with qc_output:
            qc_output.clear_output(wait=True)
            entry = qc_subjects[qc_labels.index(label)]
            d = deriv_path_for(entry)
            t1 = entry["t1w"]
            atlas_name = CONFIG["DEFAULT_ATLAS"]
            atlas_path = DERIV_DIR / f"{atlas_name}_atlas.nii.gz"

            display(HTML(
                f"<h3>{entry['subject']} "
                f"{entry['session'] or ''} — {view}</h3>"
            ))

            nv = NiiVue()
            if view == "Skull strip":
                vols = []
                if _exists(d / "N4corrected.nii.gz"):
                    vols.append({"path": str(d / "N4corrected.nii.gz")})
                elif _exists(t1):
                    vols.append({"path": str(t1)})
                if _exists(d / "BrainMask.nii.gz"):
                    vols.append({"path": str(d / "BrainMask.nii.gz"),
                                 "colormap": "red", "opacity": 0.3})
                nv.load_volumes(vols)

            elif view == "Lesion (native)":
                vols = []
                if _exists(d / "N4corrected_Brain.nii.gz"):
                    vols.append({"path": str(d / "N4corrected_Brain.nii.gz")})
                elif _exists(t1):
                    vols.append({"path": str(t1)})
                pred = d / "Prediction3_native.nii.gz"
                if _exists(pred):
                    vols.append({"path": str(pred),
                                 "colormap": "inferno", "opacity": 0.7})
                nv.load_volumes(vols)

            elif view == "Lesion (MNI)":
                vols = []
                if _exists(d / "Subject_in_MNI.nii.gz"):
                    vols.append({"path": str(d / "Subject_in_MNI.nii.gz")})
                if _exists(d / "Lesion_in_MNI.nii.gz"):
                    vols.append({"path": str(d / "Lesion_in_MNI.nii.gz"),
                                 "colormap": "inferno", "opacity": 0.7})
                nv.load_volumes(vols)

            elif view == "Lesion + atlas":
                vols = []
                if _exists(d / "Subject_in_MNI.nii.gz"):
                    vols.append({"path": str(d / "Subject_in_MNI.nii.gz")})
                if _exists(atlas_path):
                    vols.append({"path": str(atlas_path),
                                 "colormap": "viridis", "opacity": 0.4})
                else:
                    display(HTML(
                        f"<p><em>{atlas_name} atlas not yet saved to "
                        f"derivatives — run the atlas-overlap cell first.</em></p>"
                    ))
                if _exists(d / "Lesion_in_MNI.nii.gz"):
                    vols.append({"path": str(d / "Lesion_in_MNI.nii.gz"),
                                 "colormap": "red", "opacity": 0.6})
                nv.load_volumes(vols)
            display(nv)

    subj_dd = widgets.Dropdown(
        options=qc_labels, description="Subject:",
        layout=widgets.Layout(width="350px"),
        style={"description_width": "initial"},
    )
    view_tabs = widgets.ToggleButtons(
        options=["Skull strip", "Lesion (native)", "Lesion (MNI)", "Lesion + atlas"],
        description="View:",
        style={"description_width": "initial"},
    )
    display(HTML("<h3>QC dashboard</h3>"))
    display(widgets.HBox([subj_dd, view_tabs]))
    widgets.interactive_output(_show_qc, {"label": subj_dd, "view": view_tabs})
    display(qc_output)


### Group lesion overlap map

A frequency map showing, for each MNI voxel, the proportion of subjects whose
lesion includes that voxel. This is a quick way to see where lesions cluster
across the cohort.


In [ ]:
# ============================================================
# Build a group lesion-frequency map in MNI space.
# All subject lesions are resampled into a common reference (the
# first lesion that exists), binarized at CONFIG["LESION_THRESHOLD"],
# averaged, and saved to derivatives.
# ============================================================
have_lesion = [e for e in SUBJECTS if lesion_mni_for(e).exists()]

if not have_lesion:
    print("No lesion masks available yet — nothing to summarize.")
else:
    ref_img = nib.load(str(lesion_mni_for(have_lesion[0])))
    ref_shape = ref_img.shape
    accum = np.zeros(ref_shape, dtype=np.float32)
    n_used = 0
    for entry in have_lesion:
        try:
            img = nib.load(str(lesion_mni_for(entry)))
            if img.shape != ref_shape:
                img = image.resample_to_img(
                    img, ref_img, interpolation="nearest", force_resample=True,
                    copy_header=True,
                )
            data = (img.get_fdata() > CONFIG["LESION_THRESHOLD"]).astype(np.float32)
            accum += data
            n_used += 1
        except Exception as e:
            print(f"  ⚠ {entry['subject']}: {e}")

    if n_used == 0:
        print("No usable lesions — group map not built.")
    else:
        freq = accum / n_used                                  # 0..1
        freq_img = nib.Nifti1Image(freq, ref_img.affine, ref_img.header)
        out_path = DERIV_DIR / "group_lesion_frequency.nii.gz"
        nib.save(freq_img, str(out_path))
        print(f"✓ Group lesion frequency map saved to {out_path} "
              f"(n={n_used}, max overlap = {freq.max():.2f})")

        # Static glass-brain plot for a quick overview
        try:
            display = plotting.plot_glass_brain(
                freq_img, threshold=1.0 / n_used, colorbar=True,
                title=f"Lesion frequency (n={n_used})",
                cmap="hot",
            )
            plt.show()
        except Exception as e:
            print(f"  (glass-brain plot failed: {e})")

        # Interactive viewer
        try:
            from ipyniivue import NiiVue
            nv = NiiVue()
            vols = [{"path": str(out_path), "colormap": "hot", "opacity": 0.8}]
            nv.load_volumes(vols)
            from IPython.display import display as _disp
            _disp(nv)
        except Exception as e:
            print(f"  (interactive viewer failed: {e})")


### Atlas overlap

For every subject with a lesion mask, compute the overlap between the lesion and
each region of every atlas in `CONFIG["ATLASES"]`. Per-atlas results are cached
to CSV in `DERIV_DIR/lesion_atlas_overlap_<ATLAS>.csv` so the analysis only re-runs
when the source data changes.


In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y -qq ca-certificates
!sudo update-ca-certificates


In [ ]:
# ============================================================
# Compute lesion–atlas overlap for every subject and atlas.
# Results saved as one CSV per atlas in DERIV_DIR.
# ============================================================
def _atlas_arrays(atlas):
    """Pull (img, data, labels) from any nilearn-style atlas object."""
    img = atlas.maps if hasattr(atlas.maps, "get_fdata") else nib.load(atlas.maps)
    return img, img.get_fdata(), atlas.labels


def calculate_overlap(lesion_path, atlas_img, atlas_data, atlas_labels,
                      threshold=0.5):
    """Return dict[region] -> stats, plus total lesion voxel count."""
    if not Path(lesion_path).exists():
        return {}, 0
    lesion_img = nib.load(str(lesion_path))
    lesion_resampled = image.resample_to_img(
        lesion_img, atlas_img, interpolation="nearest",
        force_resample=True, copy_header=True,
    )
    lesion_binary = (lesion_resampled.get_fdata() > threshold).astype(int)
    total = int(np.sum(lesion_binary))
    if total == 0:
        return {}, 0
    overlaps = {}
    for region_idx in np.unique(atlas_data):
        if region_idx == 0:
            continue
        region_idx = int(region_idx)
        region_mask = (atlas_data == region_idx)
        n_overlap = int(np.sum(lesion_binary & region_mask))
        if n_overlap == 0:
            continue
        n_region = int(np.sum(region_mask))
        name = (atlas_labels[region_idx]
                if region_idx < len(atlas_labels) else f"Region_{region_idx}")
        overlaps[name] = {
            "lesion_in_roi_percent": 100.0 * n_overlap / total,
            "roi_coverage_percent":  100.0 * n_overlap / n_region,
            "overlap_voxels":        n_overlap,
            "total_region_voxels":   n_region,
        }
    return overlaps, total


have_lesion = [e for e in SUBJECTS if lesion_mni_for(e).exists()]
print(f"Computing overlap for {len(have_lesion)} subject(s) across "
      f"{len(ALL_ATLASES)} atlas(es)")

OVERLAP_DFS = {}                # name -> DataFrame, kept in memory
for atlas_name, atlas in ALL_ATLASES.items():
    print(f"\n--- {atlas_name} ---")
    img, data, labels = _atlas_arrays(atlas)

    # Save the atlas image once so the QC dashboard can overlay it
    atlas_path = DERIV_DIR / f"{atlas_name}_atlas.nii.gz"
    if not atlas_path.exists():
        nib.save(img, str(atlas_path))

    rows = []
    for entry in have_lesion:
        ovl, total = calculate_overlap(
            lesion_mni_for(entry), img, data, labels,
            threshold=CONFIG["LESION_THRESHOLD"],
        )
        for region, stats in ovl.items():
            rows.append({
                "subject": entry["subject"],
                "session": entry["session"] or "",
                "atlas":   atlas_name,
                "region":  region,
                **stats,
                "total_lesion_voxels": total,
            })
    df = pd.DataFrame(rows)
    OVERLAP_DFS[atlas_name] = df
    csv = DERIV_DIR / f"lesion_atlas_overlap_{atlas_name}.csv"
    df.to_csv(csv, index=False)
    print(f"  ✓ {len(df)} rows → {csv.name}")

print("\nDone.")


### Atlas overlap plots

Two views of the overlap data:

1. **Per-subject top regions** — a horizontal bar chart of the regions with the
   largest share of each subject's lesion (interactive).
2. **Cross-subject heatmap** — for the active atlas, the % of each subject's
   lesion that falls in each of the most frequently affected regions.
3. **Schaefer 7-network summary** — when Schaefer400 is in `CONFIG["ATLASES"]`,
   a stacked bar of lesion share per Yeo 7-network for every subject.


In [ ]:
# ============================================================
# Interactive atlas-overlap plots
# - Auto-updates on every widget change (no "Plot" button)
# - Works for any atlas in OVERLAP_DFS
# - Metric toggle: "lesion in ROI %"  vs  "ROI coverage %"
# - Top-N slider lets you zoom in/out
# ============================================================
import textwrap
import ipywidgets as widgets
from IPython.display import display, clear_output


METRICS = {
    "Lesion in ROI %": "lesion_in_roi_percent",
    "ROI coverage %":  "roi_coverage_percent",
}


def _wrap(label, width=38):
    """Break long atlas labels so they fit on a y-tick."""
    if isinstance(label, bytes):
        label = label.decode("utf-8", "ignore")
    return "\n".join(textwrap.wrap(str(label), width=width)) or str(label)


def _subject_label(subj, ses):
    return f"{subj} {ses}".strip() if ses else subj


def _split_label(label):
    """Reverse of _subject_label — works for labels with/without a session."""
    if " " in label:
        subj, _, ses = label.partition(" ")
        return subj, ses
    return label, ""


def _bar_chart(df, subj, ses, metric_key, metric_label, top_n):
    sel = df[(df["subject"] == subj) & (df["session"] == ses)].copy()
    if sel.empty:
        print(f"No overlap data for {subj} {ses}.")
        return
    sel = sel.sort_values(metric_key, ascending=False).head(top_n)
    # Reverse for horizontal bar (largest at top)
    sel = sel.iloc[::-1]
    labels = [_wrap(r, 38) for r in sel["region"]]
    fig, ax = plt.subplots(figsize=(9, max(2.5, 0.42 * len(sel) + 1)))
    bars = ax.barh(labels, sel[metric_key].values, color="#3b6ea8")
    for bar, value in zip(bars, sel[metric_key].values):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                f"{value:.1f}%", va="center", fontsize=8)
    ax.set_xlabel(metric_label)
    ax.set_title(f"{subj} {ses}  —  top {len(sel)} regions")
    ax.set_xlim(0, max(sel[metric_key].max() * 1.15, 1))
    ax.grid(axis="x", linestyle=":", alpha=0.5)
    plt.tight_layout()
    plt.show()


def _heatmap(df, metric_key, metric_label, top_n):
    if df.empty:
        print("No data."); return
    # Rank regions by *maximum* contribution across subjects so small
    # regions with a single strong overlap still show up.
    region_order = (df.groupby("region")[metric_key]
                      .max().sort_values(ascending=False).head(top_n).index)
    sub = df[df["region"].isin(region_order)].copy()
    sub["subj_ses"] = sub.apply(
        lambda r: _subject_label(r["subject"], r["session"]), axis=1,
    )
    pivot = (sub.pivot_table(index="region", columns="subj_ses",
                             values=metric_key, fill_value=0)
               .loc[region_order])
    if pivot.empty:
        print("No data after filtering."); return
    fig, ax = plt.subplots(
        figsize=(max(6, 0.4 * pivot.shape[1] + 4),
                 max(3, 0.35 * pivot.shape[0] + 2))
    )
    im = ax.imshow(pivot.values, aspect="auto", cmap="magma")
    ax.set_xticks(range(pivot.shape[1]))
    ax.set_xticklabels(pivot.columns, rotation=75, ha="right", fontsize=8)
    ax.set_yticks(range(pivot.shape[0]))
    ax.set_yticklabels([_wrap(r, 30) for r in pivot.index], fontsize=8)
    ax.set_title(f"{metric_label}  —  top {pivot.shape[0]} regions "
                 f"× {pivot.shape[1]} subject(s)")
    fig.colorbar(im, ax=ax, label=metric_label)
    plt.tight_layout()
    plt.show()


def _summary_table(df, subj, ses, metric_key, top_n):
    sel = df[(df["subject"] == subj) & (df["session"] == ses)].copy()
    if sel.empty:
        return "<em>No data for this subject.</em>"
    sel = sel.sort_values(metric_key, ascending=False).head(top_n)
    total = int(sel["total_lesion_voxels"].iloc[0])
    rows = "".join(
        f"<tr><td>{r.region}</td>"
        f"<td style='text-align:right'>{r.lesion_in_roi_percent:.1f}%</td>"
        f"<td style='text-align:right'>{r.roi_coverage_percent:.1f}%</td>"
        f"<td style='text-align:right'>{int(r.overlap_voxels)}</td></tr>"
        for r in sel.itertuples()
    )
    return (
        f"<p><b>Total lesion volume:</b> {total} voxels &nbsp; "
        f"<b>Regions overlapped:</b> {len(sel)}</p>"
        f"<table style='border-collapse:collapse; font-size:12px'>"
        f"<tr style='background:#2c3e50; color:white'>"
        f"<th style='padding:4px 8px; text-align:left'>Region</th>"
        f"<th style='padding:4px 8px'>Lesion%</th>"
        f"<th style='padding:4px 8px'>ROI%</th>"
        f"<th style='padding:4px 8px'>Voxels</th></tr>"
        f"{rows}</table>"
    )


if not OVERLAP_DFS:
    print("Run the atlas-overlap cell first — no data to plot.")
else:
    # ---- Widgets ----------------------------------------------
    atlas_dd = widgets.Dropdown(
        options=list(OVERLAP_DFS.keys()),
        value=(CONFIG["DEFAULT_ATLAS"] if CONFIG["DEFAULT_ATLAS"] in OVERLAP_DFS
               else list(OVERLAP_DFS.keys())[0]),
        description="Atlas:",
        layout=widgets.Layout(width="240px"),
        style={"description_width": "initial"},
    )
    subj_dd = widgets.Dropdown(
        options=[], description="Subject:",
        layout=widgets.Layout(width="300px"),
        style={"description_width": "initial"},
    )
    metric_tb = widgets.ToggleButtons(
        options=list(METRICS.keys()), description="Metric:",
        style={"description_width": "initial"},
    )
    top_n_sl = widgets.IntSlider(
        value=20, min=5, max=50, step=1, description="Top N:",
        continuous_update=False,
        layout=widgets.Layout(width="320px"),
        style={"description_width": "initial"},
    )

    def _refresh_subjects(*_):
        df = OVERLAP_DFS[atlas_dd.value]
        pairs = sorted(set(zip(df["subject"], df["session"].fillna(""))))
        labels = [_subject_label(s, ses) for s, ses in pairs]
        subj_dd.options = labels
        if labels:
            subj_dd.value = labels[0]

    atlas_dd.observe(_refresh_subjects, names="value")
    _refresh_subjects()

    bar_out   = widgets.Output()
    table_out = widgets.Output()
    heat_out  = widgets.Output()

    def _redraw(*_):
        from IPython.display import HTML
        if not OVERLAP_DFS:
            return
        df = OVERLAP_DFS[atlas_dd.value]
        metric_label = metric_tb.value
        metric_key   = METRICS[metric_label]
        subj, ses    = _split_label(subj_dd.value) if subj_dd.value else ("", "")
        top_n        = top_n_sl.value

        with bar_out:
            clear_output(wait=True)
            _bar_chart(df, subj, ses, metric_key, metric_label, top_n)
        with table_out:
            clear_output(wait=True)
            display(HTML(_summary_table(df, subj, ses, metric_key, top_n)))
        with heat_out:
            clear_output(wait=True)
            _heatmap(df, metric_key, metric_label, top_n)

    for w in (atlas_dd, subj_dd, metric_tb, top_n_sl):
        w.observe(_redraw, names="value")

    controls = widgets.VBox([
        widgets.HBox([atlas_dd, subj_dd]),
        widgets.HBox([metric_tb, top_n_sl]),
    ])

    tabs = widgets.Tab(children=[
        widgets.VBox([bar_out, table_out]),
        heat_out,
    ])
    tabs.set_title(0, "Per-subject")
    tabs.set_title(1, "Cross-subject heatmap")

    display(controls, tabs)
    _redraw()


In [ ]:
# ============================================================
# Schaefer 7-network summary (only runs if Schaefer400 was processed)
# ============================================================
NET_NAMES = ["Vis", "SomMot", "DorsAttn", "SalVentAttn", "Limbic", "Cont", "Default"]


def _net_for(label):
    """Map a Schaefer region label to one of the 7 Yeo networks."""
    if isinstance(label, bytes):
        label = label.decode("utf-8", "ignore")
    for n in NET_NAMES:
        if n in label:
            return n
    return "Other"


if "Schaefer400" not in OVERLAP_DFS or OVERLAP_DFS["Schaefer400"].empty:
    print("Schaefer400 overlap not available — skipping network summary.")
else:
    df = OVERLAP_DFS["Schaefer400"].copy()
    df["network"] = df["region"].astype(str).map(_net_for)
    summary = (df.groupby(["subject", "session", "network"])
                 ["lesion_in_roi_percent"].sum().unstack(fill_value=0))
    cols = [n for n in NET_NAMES if n in summary.columns]
    if "Other" in summary.columns:
        cols.append("Other")
    summary = summary[cols]
    fig, ax = plt.subplots(figsize=(max(6, 0.5 * len(summary) + 3), 5))
    bottom = np.zeros(len(summary))
    x = np.arange(len(summary))
    for col in summary.columns:
        ax.bar(x, summary[col].values, bottom=bottom, label=col)
        bottom += summary[col].values
    ax.set_xticks(x)
    ax.set_xticklabels([f"{s} {ses}".strip() for s, ses in summary.index],
                       rotation=90, fontsize=8)
    ax.set_ylabel("Sum of lesion-in-ROI % (Schaefer regions)")
    ax.set_title("Lesion distribution across Yeo 7 networks")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.show()


### Per-subject report (interactive + PDF)

Pick a subject and an atlas to see the full regional breakdown alongside the
lesion volume in MNI space. The same data can be exported as PDF reports — set
`GENERATE_ALL = True` to write one PDF per subject for the active atlas.


In [ ]:
# ============================================================
# Interactive per-subject report (table + viewer)
# ============================================================
from ipyniivue import NiiVue
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output


def _report_html(df, subj, ses, atlas_name):
    sel = df[(df["subject"] == subj) & (df["session"] == (ses or ""))].copy()
    if sel.empty:
        return f"<p>No data for {subj} {ses}.</p>"
    sel = sel.sort_values("lesion_in_roi_percent", ascending=False)
    total = int(sel["total_lesion_voxels"].iloc[0])
    rows = "".join(
        f"<tr><td>{r.region}</td>"
        f"<td style='text-align:right'>{r.lesion_in_roi_percent:.1f}%</td>"
        f"<td style='text-align:right'>{r.roi_coverage_percent:.1f}%</td>"
        f"<td style='text-align:right'>{int(r.overlap_voxels)}</td></tr>"
        for r in sel.itertuples()
    )
    return f"""
    <h3>{subj} {ses or ''} — {atlas_name}</h3>
    <p><b>Total lesion volume:</b> {total} voxels &nbsp;
       <b>Regions affected:</b> {len(sel)}</p>
    <table style='border-collapse:collapse; font-size:12px'>
      <tr style='background:#2c3e50; color:white'>
        <th style='padding:6px 10px; text-align:left'>Region</th>
        <th style='padding:6px 10px'>Lesion in ROI</th>
        <th style='padding:6px 10px'>ROI coverage</th>
        <th style='padding:6px 10px'>Voxels</th>
      </tr>{rows}
    </table>"""


if not OVERLAP_DFS:
    print("Run the atlas-overlap cell first.")
else:
    atlas_dd = widgets.Dropdown(
        options=list(OVERLAP_DFS.keys()), description="Atlas:",
        value=(CONFIG["DEFAULT_ATLAS"] if CONFIG["DEFAULT_ATLAS"] in OVERLAP_DFS
               else list(OVERLAP_DFS.keys())[0]),
    )
    subj_dd = widgets.Dropdown(description="Subject:", options=[])

    def _refresh_subjects(*_):
        df = OVERLAP_DFS[atlas_dd.value]
        pairs = sorted(set(zip(df["subject"], df["session"])))
        subj_dd.options = [f"{s} {ses}".strip() for s, ses in pairs]
        if subj_dd.options:
            subj_dd.value = subj_dd.options[0]

    _refresh_subjects()
    atlas_dd.observe(_refresh_subjects, names="value")

    report_out = widgets.Output()
    viz_out    = widgets.Output()

    def _show_report(*_):
        with report_out:
            clear_output()
            df = OVERLAP_DFS[atlas_dd.value]
            label = subj_dd.value
            subj, _, ses = label.partition(" ")
            display(HTML(_report_html(df, subj, ses, atlas_dd.value)))
        with viz_out:
            clear_output()
            label = subj_dd.value
            subj, _, ses = label.partition(" ")
            entry = next((e for e in SUBJECTS
                          if e["subject"] == subj
                          and (e["session"] or "") == ses), None)
            if entry is None:
                return
            d = deriv_path_for(entry)
            atlas_path = DERIV_DIR / f"{atlas_dd.value}_atlas.nii.gz"
            vols = []
            if (d / "Subject_in_MNI.nii.gz").exists():
                vols.append({"path": str(d / "Subject_in_MNI.nii.gz")})
            if atlas_path.exists():
                vols.append({"path": str(atlas_path),
                             "colormap": "viridis", "opacity": 0.4})
            if (d / "Lesion_in_MNI.nii.gz").exists():
                vols.append({"path": str(d / "Lesion_in_MNI.nii.gz"),
                             "colormap": "red", "opacity": 0.6})
            nv = NiiVue()
            nv.load_volumes(vols)
            display(nv)

    atlas_dd.observe(_show_report, names="value")
    subj_dd.observe(_show_report, names="value")

    display(HTML("<h3>Interactive lesion report</h3>"))
    display(widgets.HBox([atlas_dd, subj_dd]))
    display(viz_out)
    display(report_out)
    _show_report()


In [ ]:
# ============================================================
# PDF report generator — uses the in-memory OVERLAP_DFS
# ============================================================
from matplotlib.backends.backend_pdf import PdfPages

GENERATE_ALL = False                                  # set True for batch
PDF_ATLAS    = CONFIG["DEFAULT_ATLAS"]
PDF_SUBJECT  = None                                   # None = first available
PDF_SESSION  = None                                   # None = first available

REPORTS_DIR = DERIV_DIR / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)


def write_pdf(subj, ses, df, atlas_name):
    sel = df[(df["subject"] == subj) & (df["session"] == (ses or ""))].copy()
    if sel.empty:
        print(f"  no data for {subj} {ses} ({atlas_name})"); return None
    sel = sel.sort_values("lesion_in_roi_percent", ascending=False)
    total = int(sel["total_lesion_voxels"].iloc[0])
    fname = REPORTS_DIR / f"{subj}_{ses or 'no-ses'}_{atlas_name}.pdf"

    with PdfPages(str(fname)) as pdf:
        # Page 1 — summary + bar chart
        fig = plt.figure(figsize=(8.5, 11))
        plt.suptitle(f"Lesion report  —  {subj} {ses or ''}  ({atlas_name})",
                     fontsize=14, fontweight="bold")
        ax_meta = fig.add_axes([0.08, 0.78, 0.84, 0.12]); ax_meta.axis("off")
        ax_meta.text(0, 0.6,
            f"Date: {datetime.now():%Y-%m-%d %H:%M}\n"
            f"Total lesion volume: {total} voxels\n"
            f"Regions affected: {len(sel)}",
            family="monospace", fontsize=10)
        ax_bar = fig.add_axes([0.08, 0.08, 0.84, 0.62])
        top = sel.head(20)
        ax_bar.barh(top["region"][::-1], top["lesion_in_roi_percent"][::-1])
        ax_bar.set_xlabel("% of lesion in ROI")
        ax_bar.set_title(f"Top {len(top)} affected regions")
        pdf.savefig(fig); plt.close(fig)

        # Subsequent pages — full table
        rows_per_page = 35
        for start in range(0, len(sel), rows_per_page):
            chunk = sel.iloc[start:start + rows_per_page]
            fig = plt.figure(figsize=(8.5, 11))
            plt.suptitle(f"All regions  —  rows {start + 1}-{start + len(chunk)} "
                         f"of {len(sel)}", fontsize=11)
            ax = fig.add_axes([0.05, 0.05, 0.9, 0.85]); ax.axis("off")
            text = (f"{'Rank':<5s}{'Region':<42s}"
                    f"{'Lesion%':>10s}{'ROI%':>8s}{'vox':>8s}\n"
                    + "-" * 75 + "\n")
            for i, row in enumerate(chunk.itertuples(), start + 1):
                text += (f"{i:<5d}{row.region[:40]:<42s}"
                         f"{row.lesion_in_roi_percent:>9.1f}%"
                         f"{row.roi_coverage_percent:>7.1f}%"
                         f"{int(row.overlap_voxels):>8d}\n")
            ax.text(0, 1, text, family="monospace", fontsize=8,
                    verticalalignment="top")
            pdf.savefig(fig); plt.close(fig)
    print(f"  ✓ {fname}")
    return fname


df = OVERLAP_DFS.get(PDF_ATLAS)
if df is None or df.empty:
    print(f"No data for atlas '{PDF_ATLAS}'.")
else:
    targets = (sorted(set(zip(df["subject"], df["session"])))
               if GENERATE_ALL
               else [(PDF_SUBJECT or df["subject"].iloc[0],
                      PDF_SESSION or df["session"].iloc[0])])
    print(f"Writing {len(targets)} PDF(s) to {REPORTS_DIR}")
    for s, sess in targets:
        write_pdf(s, sess, df, PDF_ATLAS)


## Summary

A snapshot of how the run went — what was discovered, segmented, analysed, and
produced. Useful when re-running on a new dataset.


In [ ]:
# ============================================================
# Final summary
# ============================================================
n_total      = len(SUBJECTS)
n_lesion     = sum(1 for e in SUBJECTS if lesion_mni_for(e).exists())
atlas_csvs   = list(DERIV_DIR.glob("lesion_atlas_overlap_*.csv"))
group_map    = DERIV_DIR / "group_lesion_frequency.nii.gz"
report_files = list((DERIV_DIR / "reports").glob("*.pdf")) if (DERIV_DIR / "reports").exists() else []

print("=" * 60)
print("LINDA notebook run summary")
print("=" * 60)
print(f"Project dir         : {PROJECT_DIR}")
print(f"Dataset source      : {CONFIG['DATASET_SOURCE']}")
print(f"Subjects discovered : {n_total}")
print(f"With lesion masks   : {n_lesion}")
print(f"Atlases processed   : {len(atlas_csvs)}  "
      f"({', '.join(c.stem.replace('lesion_atlas_overlap_', '') for c in atlas_csvs)})")
print(f"Group lesion map    : {'yes' if group_map.exists() else 'no'}")
print(f"PDF reports written : {len(report_files)}")
print()
print(f"All outputs are under: {DERIV_DIR}")
